# 📊 Análisis de Docentes — IE Públicas del Perú

**Dataset:** Docentes por Institución Educativa Pública  
**Fuente:** Ministerio de Educación del Perú (MINEDU)  
**Autor:** Portfolio de Análisis de Datos  

---

## Tabla de Contenidos
1. [Setup e Importaciones](#1-setup)
2. [Pipeline ETL](#2-etl)
3. [Exploración del Dataset (EDA)](#3-eda)
4. [Consultas SQL](#4-sql)
5. [Visualizaciones](#5-visualizaciones)
6. [Conclusiones](#6-conclusiones)

## 1. Setup e Importaciones

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import warnings
warnings.filterwarnings('ignore')

from src.etl     import extract, transform, load
from src.queries import *

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')
PALETTE = ['#2563EB','#10B981','#F59E0B','#EF4444','#8B5CF6','#EC4899']

print('Setup completado ✓')

## 2. Pipeline ETL

In [ ]:
# EXTRACT
df_raw = extract('../data/raw/docentes_raw.csv')
print(f'Shape raw: {df_raw.shape}')

In [ ]:
# TRANSFORM
df = transform(df_raw)
df.head(3)

In [ ]:
# LOAD
conn = load(df, csv_path='../data/processed/docentes_clean.csv',
               db_path='../data/processed/docentes.db')
print('Datos cargados en SQLite ✓')

## 3. Exploración del Dataset (EDA)

In [ ]:
print('=== Info General ===')
print(f'Registros    : {len(df):,}')
print(f'Columnas     : {df.shape[1]}')
print(f'Nulos totales: {df.isnull().sum().sum()}')
print(f'Departamentos: {df["dpto"].nunique()}')
print(f'Niveles      : {df["nivel_clean"].unique()}')

In [ ]:
# Estadísticas descriptivas — variables clave
cols = ['Docentes_total','pct_mujeres','pct_nombrados','pct_con_titulo']
df[cols].describe().round(2)

In [ ]:
# Distribución por área
df['area_clean'].value_counts(normalize=True).mul(100).round(1)

## 4. Consultas SQL

In [ ]:
# KPIs nacionales
resumen_nacional(conn)

In [ ]:
# Top 10 departamentos
top_departamentos(conn, n=10)

In [ ]:
# Distribución por nivel
distribucion_por_nivel(conn)

In [ ]:
# Brecha urbana/rural
indicadores_por_area(conn)

In [ ]:
# Ranking sin título
ranking_sin_titulo(conn, n=10)

## 5. Visualizaciones

In [ ]:
# Fig 1 - Top departamentos
from src.visualizations import fig_top_departamentos
fig_top_departamentos(top_departamentos(conn))
from IPython.display import Image
Image('../outputs/figures/01_top_departamentos.png')

In [ ]:
# Fig 2 - Nivel educativo
from src.visualizations import fig_nivel_educativo
fig_nivel_educativo(distribucion_por_nivel(conn))
Image('../outputs/figures/02_nivel_educativo.png')

In [ ]:
# Fig 3 - Urbano vs Rural
from src.visualizations import fig_urbano_rural
fig_urbano_rural(indicadores_por_area(conn))
Image('../outputs/figures/03_urbano_rural.png')

In [ ]:
# Fig 4 - Sin título
from src.visualizations import fig_sin_titulo
fig_sin_titulo(ranking_sin_titulo(conn))
Image('../outputs/figures/04_sin_titulo.png')

In [ ]:
# Fig 5 - Pirámide etaria
from src.visualizations import fig_piramide_etaria
fig_piramide_etaria(piramide_etaria_nacional(conn))
Image('../outputs/figures/05_piramide_etaria.png')

In [ ]:
# Fig 6 - Correlación
from src.visualizations import fig_correlacion
fig_correlacion(df)
Image('../outputs/figures/06_correlacion.png')

In [ ]:
# Fig 7 - Boxplot
from src.visualizations import fig_boxplot_nivel_area
fig_boxplot_nivel_area(df)
Image('../outputs/figures/07_boxplot_nivel_area.png')

In [ ]:
# Fig 8 - Scatter
from src.visualizations import fig_scatter_titulos
fig_scatter_titulos(df)
Image('../outputs/figures/08_scatter_titulos.png')

## 6. Conclusiones

| Indicador | Valor Nacional |
|---|---|
| Total instituciones | 66,052 |
| Total docentes | 405,024 |
| % Mujeres | 61.6% |
| % Docentes nombrados | 65.8% |
| % Docentes titulados | 96.4% |
| % IE rurales | 77.0% |

### Hallazgos principales
- **Feminización marcada:** 6 de cada 10 docentes son mujeres.
- **Alta titulación, brechas regionales:** Loreto y Ucayali superan el 10% sin título.
- **Predominancia rural con menor estabilidad:** el 77% de las IE son rurales pero tienen menor % de nombrados.
- **Fuerza laboral de mediana edad:** el grupo 36-45 es el más numeroso; el relevo generacional es un riesgo futuro.


In [ ]:
conn.close()
print('Análisis completado ✓')